# 04 — Phân tách QRS và T loops trực tiếp từ Frank VCG

Notebook dùng module đi kèm, chỉ đọc `vx/vy/vz`. Mốc căn nhịp là **đỉnh QRS trên VCG**, không phải R của ECG lead II. Mục tiêu là giữ cấu trúc QRS/T, tạo hai quỹ đạo XYZ và 12 đặc trưng có thể kiểm tra bằng mắt.

`XYZ → nhánh detector → mốc QRS → cắt/căn beat → median beat → biên QRS/T → loops → PCA/12 features`

**Bản này là một implementation nghiên cứu có QC, không phải thuật toán đã được xác thực là tốt nhất.** PTB trong workspace không có annotation biên QRS/T được dùng làm ground truth. Chạy không lỗi, biên hợp lệ và hình đẹp không phải độ chính xác phân đoạn. Cần kiểm tra bằng mắt/annotation chuyên gia trước khi dùng làm kết quả nghiên cứu.

Điểm khác notebook 03: không làm mượt XYZ trực tiếp với 1201 mẫu; nhánh band-pass chỉ dùng phát hiện nhịp; gốc loop được đặt từ PQ; biên không rõ được gắn cờ hoặc để thiếu. Notebook 03 và các CSV cũ không bị sửa.

Phiên bản tiền xử lý có điều kiện và QC dùng chung module với notebook 05; các cấu hình nghiên cứu được kiểm chứng bằng đáp ứng và thí nghiệm, không được coi là tối ưu y khoa.

## Chuẩn bị dữ liệu trên máy này
Chạy cell dưới một lần mỗi phiên. Dữ liệu PTB được tải từ GitHub Releases nếu còn thiếu.
Notebook 06–09 tự tạo các đầu vào phụ thuộc còn thiếu; lần đầu có thể chạy lâu.
Hướng dẫn và dung lượng: xem `README.md` và `docs/REPRODUCIBILITY.md`.

In [ ]:
from project_setup import prepare_notebook
prepare_notebook(4)

## 1. Thư viện và cấu hình

Chạy từ thư mục project, chọn kernel có NumPy, SciPy, pandas, Matplotlib, scikit-learn và WFDB. Cài đúng phiên bản bằng `python -m pip install -r requirements.txt` theo README, chọn kernel **PTB VCG** rồi restart kernel.

Mặc định xử lý 30 giây đầu, vẽ 3 record demo ; batch MI/HC mặc định tắt cho kiểm tra nhanh. Đặt `RUN_BATCH=False` để chỉ xem demo. Các output mới nằm trong `vcg_preprocessing_batch_results/`.

In [ ]:
# %pip install numpy scipy pandas matplotlib scikit-learn wfdb
from pathlib import Path
from dataclasses import asdict, replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, savgol_filter, butter, sosfiltfilt
from scipy.ndimage import uniform_filter1d, percentile_filter, binary_closing
from sklearn.decomposition import PCA
from vcg_preprocessing import (Config, legacy_config, design_morphology_fir,
    analyze_powerline_noise, decide_notch, assess_input_quality,
    filter_frequency_response, welch_xyz)
from vcg_pipeline import (FEATURE_NAMES, preprocess_xyz, ms_samples, odd_window, robust_sigma, load_xyz, detect_qrs_anchors, spatial_speed, flattest_pq, representative_beat, quiet_boundary, delineate_loops, apply_manual_boundaries, cut_loop, project_pca, extract_features, process_record, qc_row, plot_detection, plot_delineation, plot_loops)
from vcg_preprocessing_evaluation import (compare_preprocessing_configs,
    powerline_diagnostic_table, plot_filter_responses, plot_preprocessing_diagnostics,
    plot_median_config_comparison, plot_3d_config_comparison)
CFG=Config()


In [ ]:
import json
from IPython.display import display

DATA_ROOT = Path('content/ptbdb/physionet.org/files/ptbdb/1.0.0')
OUTPUT_DIR = Path('vcg_preprocessing_batch_results')
DEMO_RECORDS = ['patient104/s0306lre', 'patient001/s0010_re', 'patient002/s0015lre']
RUN_BATCH = False
BATCH_LIMIT = None  # None = tất cả MI/HC; ví dụ 20 để chạy nhanh
MANUAL_BOUNDARIES_MS = {}  # record_id -> dict qrs_on/qrs_off/t_on/t_off, đơn vị ms

if not DATA_ROOT.is_dir():
    raise FileNotFoundError('Hãy mở notebook từ thư mục project hoặc sửa DATA_ROOT.')
plt.rcParams.update({'font.family': 'DejaVu Sans', 'figure.dpi': 100})
display(pd.Series(asdict(CFG), name='Cấu hình'))

## 2. Tiền xử lý dùng chung với notebook 05

FIR 401/Kaiser8/cutoff175 tại 1000 Hz, Notch auto dựa trên PSD/persistence và vùng ít hoạt động, Q50 khi được áp dụng. Morphology không mặc định high-pass; detector giữ Butterworth bậc 3 5–30 Hz. `preprocess_xyz` vẫn trả hai mảng mặc định; `return_diagnostics=True` mở thêm dictionary. Xem các hình kiểm chứng và A/B/C trong notebook 05.

In [ ]:
# Hàm dùng chung được import ở ô thư viện; xem module để đọc implementation.

## 3. Tìm nhịp trực tiếp trên VCG

Detector tạo envelope RMS từ tổng bình phương ba kênh đã band-pass, tích phân ngắn 80 ms. Ngưỡng thay đổi theo nền và biên độ trong cửa sổ 2 giây. Candidate cách nhau ít nhất 250 ms, rồi được tinh chỉnh thành cực đại magnitude XYZ trong ±60 ms. Refractory được áp dụng lại sau tinh chỉnh.

Đây là detector tự triển khai, không phải Pan–Tompkins đầy đủ hay bản sao BRAVEHEART. Đỉnh QRS có thể lệch mốc R của một lead ECG. Detector không thể tự bảo đảm phân biệt mọi T cao, QRS phân mảnh, nhiễu hoặc nhịp ngoại tâm thu. RR bất thường được lưu thành cờ; không tự thêm nhịp bị nghi là thiếu.

In [ ]:
# Hàm dùng chung được import ở ô thư viện; xem module để đọc implementation.

## 4. Căn baseline và tạo median beat của nhịp chiếm ưu thế

Cửa sổ beat phụ thuộc median RR: trước mốc tối đa 280 ms, sau mốc tối đa 650 ms. Loại beat sát biên, RR quá khác median hoặc nguy cơ lẫn QRS kế tiếp. Trong mỗi beat, chọn đoạn PQ ít biến thiên nhất trước QRS, rồi trừ median XYZ của đoạn đó.

Các beat có tương quan QRS dưới 0,85 được gắn cờ khác hình thái và giữ lại nếu không có lỗi dữ liệu rõ. Không coi hình thái MI khác là nhiễu. Median theo từng thời điểm/trục tạo một beat đại diện; không co giãn thời gian, để giữ đơn vị vận tốc. `beat_qc` ghi lý do giữ/loại. Đây là cách đại diện **nhịp chiếm ưu thế**, không mô tả đầy đủ ngoại tâm thu/nhịp không đều. PQ phẳng không tự động có nghĩa PQ đúng; vùng xám trên hình giúp kiểm tra điều này.

In [ ]:
# Hàm dùng chung được import ở ô thư viện; xem module để đọc implementation.

## 5. Ước lượng biên QRS và T

**QRS:** tốc độ `||dXYZ/dt||` được tính bằng đạo hàm Savitzky–Golay ngắn 11 ms. Tìm cụm hoạt động quanh mốc QRS trong −180 đến +200 ms. Ngưỡng là max(nền + 4 MAD chuẩn hóa, 7% đỉnh tốc độ). Nối khe hoạt động ngắn 15 ms để hạn chế cắt giữa Q/R/S; tìm đoạn yên 12 ms ở hai phía. Không tìm được biên thì trả `None`, không mặc định lấy biên cửa sổ.

**T:** tìm cực đại rõ của magnitude sau QRS; cửa sổ kết thúc trước QRS kế tiếp theo RR. Mốc T-on được định nghĩa thao tác bằng tiếp tuyến sườn lên cắt mức ST; T-end bằng tiếp tuyến sườn xuống cắt mức TP sau T. Mức TP lấy từ đoạn phẳng 25 ms sau đỉnh T. Có thêm biên theo ngưỡng để kiểm tra chéo. Không đủ tiếp tuyến, baseline hoặc biên không có thứ tự hợp lệ sẽ gắn cờ; không ép một độ dài T tối thiểu để tạo vòng giả.

**T-on không có một định nghĩa duy nhất trong mọi hình thái.** Quy tắc tiếp tuyến ở đây có thể chọn khác quy tắc ngưỡng hoặc cách lấy toàn ST–T; điều này thay đổi các feature T. ST khác nền PQ, nhiều lobe T, TP không rõ và bất đồng phương pháp cần kiểm tra thủ công. Mốc tiếp tuyến là quy ước đo, không phải điểm điện thế đã hoàn toàn về 0.

In [ ]:
# Hàm dùng chung được import ở ô thư viện; xem module để đọc implementation.

## 6. Cắt loop, PCA và 12 feature

`qrs_loop = XYZ[QRS_on:QRS_off+1]`; T tương tự. Vận tốc dùng cùng đạo hàm ngắn trên toàn median beat rồi mới lấy đoạn để giảm ảnh hưởng đạo hàm tại mép đoạn.

PCA fit **riêng** cho từng loop để tính Arc/Area. Tâm PCA gần (0,0) là tâm đã trừ, khác gốc PQ trong XYZ. Hai PCA riêng không thể dùng để đo vị trí tương đối hoặc góc QRS–T. Arc là đường mở theo thời gian; Area dùng shoelace đóng điểm cuối về điểm đầu. Với đường tự cắt, shoelace là trị tuyệt đối diện tích đại số, không bảo đảm bằng toàn phần vùng tô.

7 QRS: Arc, Area, MaxVec, MaxCentroidDistance, Max/Mean/StdVel. 5 T: Area, MaxVec, Max/Mean/StdVel. Đơn vị Arc/MaxVec/khoảng cách là mV, Area mV², velocity mV/s. Feature là candidate và phải đi kèm QC.

In [ ]:
# Hàm dùng chung được import ở ô thư viện; xem module để đọc implementation.

## 7. Các biểu đồ kiểm tra

1. **Recording:** XYZ gốc và morphology trên cùng trục; các đường đứng là mốc phát hiện trực tiếp từ VCG. Dùng hình này để kiểm tra đếm nhịp và ảnh hưởng lọc.
2. **Median beat:** vùng xanh là candidate QRS, cam là candidate T; hình vận tốc giải thích QRS, hình magnitude có tiếp tuyến giải thích T. Vùng xám là baseline PQ, vùng tím nhạt là TP dùng cho T-end.
3. **Loops:** XYZ 3D và Frank X–Y dùng cùng hệ tọa độ; PCA QRS/T được vẽ ở hai ô riêng. `o` là đầu đoạn, `×` là cuối. Khi so kích thước giữa record, đọc số và giới hạn trục; các hình không có cùng thang đo mặc định.

Một vùng tô màu không chứng minh biên đúng. `auto_candidate` chỉ nghĩa không vi phạm các kiểm tra heuristic được cài đặt.

In [ ]:
# Hàm dùng chung được import ở ô thư viện; xem module để đọc implementation.

## 8. Chạy ba record demo và xem kết quả

HC patient104 và MI patient001 là các record từng dùng trong notebook 03; patient002 là một MI khác. Nhãn diagnosis chỉ để mô tả, không dùng để chỉnh ngưỡng hoặc chọn biên. Các record demo không đủ để kết luận feature nào phân biệt bệnh.

`features` vẫn hiển thị các giá trị candidate của ca cần review, nhưng batch chỉ đưa ca qua QC vào bảng `features_qc_pass.csv`. Ca thiếu biên T sẽ có feature T là NaN.

In [ ]:
demo_results = {}
for record_id in DEMO_RECORDS:
    print()
    print(record_id)
    try:
        result = process_record(DATA_ROOT / record_id, CFG,
                                manual_ms=MANUAL_BOUNDARIES_MS.get(record_id))
        demo_results[record_id] = result
        display(pd.DataFrame([qc_row(result)]))
        for fn in [plot_detection, plot_delineation, plot_loops]:
            fn(result)
            plt.show()
            plt.close()
        display(pd.Series(result['features'], name='Candidate features').to_frame())
    except (ValueError, OSError) as exc:
        print(f'Không đủ dữ liệu/không đọc được record: {type(exc).__name__}: {exc}')

## 9. Xem bất kỳ record và chỉnh biên sau khi kiểm tra

`inspect_record()` trả toàn bộ arrays, detector, beat QC, biên và feature. Nếu một biên chưa rõ, xem cả hình recording và median beat. Sửa biên bằng `manual_ms` chỉ sau khi đã kiểm tra; các mốc tính theo mốc QRS VCG tại 0 ms. Hàm kiểm tra thứ tự và giới hạn mảng, không âm thầm clip biên.

Để dùng mốc đã kiểm tra trong batch, thêm vào `MANUAL_BOUNDARIES_MS[record_id]` rồi chạy lại batch. Việc sửa biên không tự xóa cờ lỗi detector/nhịp không đều. `manual_reviewed` chỉ ghi nguồn biên thủ công, không phải chứng nhận chuyên gia.

In [ ]:
def inspect_record(record_id='patient002/s0015lre', manual_ms=None):
    result = process_record(DATA_ROOT / record_id, CFG, manual_ms=manual_ms)
    display(pd.DataFrame([qc_row(result)]))
    for fn in [plot_detection, plot_delineation, plot_loops]:
        fn(result); plt.show(); plt.close()
    display(result['rep']['beat_qc'])
    return result

# Chạy thử record khác:
# result = inspect_record('patient003/s0017lre')
# Mốc đã kiểm tra: dict có đúng qrs_on, qrs_off, t_on, t_off (ms).
# MANUAL_BOUNDARIES_MS['patient.../record...'] = your_reviewed_times_ms
# result = inspect_record('patient.../record...', manual_ms=your_reviewed_times_ms)

## 10. Batch MI/HC và xuất kết quả mới

Mapping diagnosis exact-match sau chuẩn hóa. Metadata chỉ tạo nhãn/truy vết, không là input detector. `failures.csv` lưu record không tạo được median beat hoặc lỗi đọc; `qc.csv` chứa mọi record, kể cả lỗi; `beat_qc.csv` ghi lý do giữ/loại từng nhịp.

`candidate_features.csv` có tất cả candidate kèm status/flags; `features_qc_pass.csv` chỉ có ca qua kiểm tra nội bộ. **Không dùng số ca qua QC làm độ chính xác.** Việc loại ca ST chênh hoặc nhịp bất thường có thể thay đổi tỷ lệ MI/HC và gây selection bias; phải báo cáo tỷ lệ theo nhóm và review ca bị loại trước khi huấn luyện. Đây chưa phải một cohort đã được xác thực.

In [ ]:
def scan_mi_hc(data_root):
    rows = []
    for hea in sorted(Path(data_root).rglob('*.hea')):
        diagnosis = next((line.split(':', 1)[1].strip() for line in hea.read_text(encoding='latin-1').splitlines()
                          if line.strip().lower().startswith('# reason for admission:')), '')
        norm = ' '.join(diagnosis.lower().split())
        label = 'MI' if norm == 'myocardial infarction' else ('HC' if norm in {'healthy control', 'healthy controls'} else None)
        if label:
            record_path = hea.with_suffix('')
            rows.append({'record_path': record_path, 'record_id': record_path.relative_to(data_root).as_posix(),
                         'patient_id': record_path.parent.name, 'label': label})
    return rows


def run_batch(data_root=DATA_ROOT, output_dir=OUTPUT_DIR, limit=BATCH_LIMIT):
    records = scan_mi_hc(data_root)
    if limit is not None: records = records[:limit]
    if not records: raise ValueError('Không tìm thấy record MI/HC trong DATA_ROOT.')
    qc_rows, candidates, beat_rows, failures = [], [], [], []
    for i, meta in enumerate(records, 1):
        identity = {k:meta[k] for k in ['record_id', 'patient_id', 'label']}
        try:
            result = process_record(meta['record_path'], CFG,
                                    manual_ms=MANUAL_BOUNDARIES_MS.get(meta['record_id']))
            row = {**qc_row(result), **identity}
            qc_rows.append(row)
            candidates.append({**identity, 'status':result['status'],
                               'eligible_for_features':result['eligible_for_features'],
                               'flags':'; '.join(result['flags']), **result['features']})
            for beat in result['rep']['beat_qc'].to_dict('records'):
                beat_rows.append({**identity, **beat})
        except Exception as exc:
            row = {**identity, 'status':'failed', 'eligible_for_features':False,
                   'error':f'{type(exc).__name__}: {exc}'}
            qc_rows.append(row); failures.append(row)
        if i % 100 == 0 or i == len(records):
            print(f'{i}/{len(records)} records; không tạo được kết quả: {len(failures)}')
    qc = pd.DataFrame(qc_rows)
    candidate_cols = ['record_id','patient_id','label','status','eligible_for_features','flags',*FEATURE_NAMES]
    candidate = pd.DataFrame(candidates, columns=candidate_cols)
    passed = candidate[candidate['eligible_for_features'] == True].copy()
    output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    qc.to_csv(output_dir / 'qc.csv', index=False)
    candidate.to_csv(output_dir / 'candidate_features.csv', index=False)
    passed.to_csv(output_dir / 'features_qc_pass.csv', index=False)
    pd.DataFrame(beat_rows).to_csv(output_dir / 'beat_qc.csv', index=False)
    pd.DataFrame(failures, columns=['record_id','patient_id','label','status','eligible_for_features','error']).to_csv(output_dir / 'failures.csv', index=False)
    (output_dir / 'config.json').write_text(json.dumps({'config':asdict(CFG),
        'manual_boundaries_ms':MANUAL_BOUNDARIES_MS, 'n_records':len(records),
        'method':'VCG RMS detector; PQ baseline; speed QRS; tangent T with TP baseline',
        'ground_truth_validation':False}, ensure_ascii=False, indent=2), encoding='utf-8')
    return qc, candidate, passed


if RUN_BATCH:
    df_qc_v4, df_candidates_v4, df_qc_pass_v4 = run_batch()
    display(pd.crosstab(df_qc_v4['label'], df_qc_v4['status']))
    display(df_qc_v4[df_qc_v4['status'] != 'auto_candidate'].head(15))
    print(f'Candidate feature rows: {len(df_candidates_v4)}; qua QC nội bộ: {len(df_qc_pass_v4)}')
    print('Đây là kết quả kiểm tra xử lý, không phải accuracy so với annotation.')
    print('Output:', OUTPUT_DIR.resolve())
else:
    print('RUN_BATCH=False; chỉ chạy các record demo.')

## 11. Tài liệu tham khảo và phạm vi kiểm chứng

- [BRAVEHEART, Stabenau & Waks, 2023](https://doi.org/10.1016/j.cmpb.2023.107798): tham khảo phát hiện QRS trên magnitude VCG và quy trình có kiểm tra chất lượng; notebook này không tái hiện nguyên thuật toán hoặc mạng neural của BRAVEHEART.
- [Kijonka et al., 2022 — A wavelet-based VCG QRS loop boundaries and isoelectric coordinates detector](https://doi.org/10.3389/fphys.2022.941827): tham khảo phát hiện QRS/PQ trực tiếp trên VCG; notebook này dùng energy/speed, không triển khai bộ wavelet của paper và không thừa hưởng số liệu độ chính xác của paper.
- [Ví dụ đo T-end trên VCG bằng tiếp tuyến trong nghiên cứu dùng LEADS](https://pure.uva.nl/ws/files/43095264/Chapter_8.pdf): cơ sở tham khảo cho T-end trên magnitude; onset tiếp tuyến và các ngưỡng cụ thể trong notebook là lựa chọn triển khai của nhóm.
- [PCA của scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html): PCA trừ tâm, giữ hai thành phần; `explained_variance_ratio_` mô tả biến thiên được giữ, không phải độ chính xác segmentation.

Kiểm tra phần mềm gồm tín hiệu tổng hợp ở 250/500/1000 Hz, ca không có T, tín hiệu phẳng, tính đơn vị/scaling feature và chạy batch PTB. Tín hiệu tổng hợp chỉ kiểm tra hành vi có kiểm soát; PTB batch chỉ kiểm tra khả năng chạy và cờ QC. Đánh giá thật cần annotation QRS_on/off và T_on/off độc lập, báo sai số từng biên và tỷ lệ không phân đoạn được. Không suy diễn tính đúng lâm sàng từ bảng QC.

Sau khi kiểm tra và chốt phương pháp, có thể dùng `process_record()` thay bước xử lý record trong notebook 02. Bộ feature dùng median beat và bộ lọc mới, nên cần trích xuất lại toàn cohort trước khi so sánh/huấn luyện; không trộn với CSV tạo bằng pipeline cũ.